# Data QA + DBSCAN Sizing

**Purpose:** the data-engineering checklist in `paper/TASKS.md` Section A — sort/clean GPS
pings with per-ping motion features (dt/dist/speed, implausible-jump flagging), QA the
mine-drawn zone polygons against actual GPS stop density, and use real zone sizes to pick a
DBSCAN `eps`/`min_samples` range for the stop-detection pipeline (Section C).

In [ ]:
import sys
sys.path.append("../..")

from gps_lib import io_utils, classify, preprocess, zones

## 1. Load + clean GPS pings

In [ ]:
tracker_list_df = io_utils.load_tracker_list()
tracker_list_df = classify.classify_technic_material_type(tracker_list_df)

track_points_df = io_utils.load_gps_data()
merged_df = preprocess.attach_technic_info(track_points_df, tracker_list_df)
df_cleaned = preprocess.clean_gps_points(merged_df, round_n=4)
df_cleaned.shape

## 2. Per-ping motion features (dt / dist / speed) + implausible-jump flags

`add_motion_features` sorts by `tracker_id, get_time`, then computes seconds since the
previous ping, haversine distance from the previous ping, implied speed, and flags pings
implying speed above `max_speed_kmh` as likely GPS glitches.

In [ ]:
motion_df = preprocess.add_motion_features(df_cleaned, max_speed_kmh=120.0)

n_flagged = motion_df["implausible_jump"].sum()
print(f"Implausible-speed pings: {n_flagged:,} / {len(motion_df):,} "
      f"({n_flagged / len(motion_df):.3%})")
motion_df["speed_kmh"].describe()

In [ ]:
# Worst offenders — useful to eyeball before deciding whether to drop or just flag them
motion_df.sort_values("speed_kmh", ascending=False)[
    ["tracker_id", "get_time", "dt", "dist", "speed_kmh", "implausible_jump"]
].head(20)

## 3. Zone polygon QA

Build the zone geometry, then check whether trucks actually stop inside each mine-drawn
zone polygon. Zones with ~0 stopped pings despite being labeled load/dump/fuel/repair are
candidates for stale or mis-drawn geometry.

In [ ]:
zone_list_df = io_utils.load_zone_list()
zone_list_df = classify.classify_zones(zone_list_df, drop_other=True)
zone_detail_all_df = io_utils.load_zone_detail()

zones_gdf = zones.build_zone_geodataframe(zone_detail_all_df, zone_list_df)
zones_gdf.shape

In [ ]:
density_df = zones.zone_ping_density(motion_df, zones_gdf, speed_col="speed", speed_threshold=2.0)
print("Zones with zero stopped pings (QA flag):", (density_df["n_stopped_pings"] == 0).sum())
density_df.head(15)

## 4. Zone size → DBSCAN eps/min_samples recommendation

Compute each zone's bounding-box diagonal (meters). `eps` should be small relative to the
*smallest* real zones (so adjacent zones don't merge into one cluster), and `min_samples`
should reflect the ping rate (~10-30s interval) over a plausible minimum dwell time.

In [ ]:
diam_df = zones.zone_diameter_stats(zones_gdf)
diam_df["diameter_m"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

In [ ]:
smallest = diam_df.head(10)
print("Smallest zones (tightest constraint on eps):")
smallest[["label", "zone_material_type", "diameter_m"]]

**Recommendation:** set DBSCAN `eps` to roughly half the smallest real zone diameter
(`diam_df["diameter_m"].quantile(0.1) / 2`, printed above) so clusters stay within a single
zone's footprint, and set `min_samples` to the number of pings expected during a minimum
plausible dwell (e.g. ~60s dwell / ~20s ping interval ≈ 3). Tune both against the
parameter-sensitivity sweep in `paper/TASKS.md` Section C once the DBSCAN pipeline exists —
treat these as a starting search range, not a final answer.